# 07 — Image & Bib Attribute Analysis

Correlates image and bib-level attributes (brightness, contrast, sharpness, size, frame position)
with model outcomes (TP / FN / confidence) for both YOLOv8n and RF-DETR Nano.

**Goal:** Identify which image conditions lead to missed detections and where the training
data has gaps — giving concrete direction for future data collection.

Each row of the core DataFrame (`bib_df`) represents one ground-truth bib in the test set,
with columns for every attribute and for each model's outcome.

## Setup

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import torch
from PIL import Image
from rfdetr import RFDETRNano
from ultralytics import YOLO

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
ROOT = Path("..")

DATASET_DIR = ROOT / "data" / "labeled" / "race-vision.v1i.yolov8"

# Which splits to run inference on and include in the analysis.
# Use ["test"] for a quick run; ["train", "val", "test"] for full dataset diagnostics.
EVAL_SPLITS = ["train", "val", "test"]

SPLIT_DIRS = {
    "train": (DATASET_DIR / "train" / "images", DATASET_DIR / "train" / "labels"),
    "val":   (DATASET_DIR / "valid" / "images", DATASET_DIR / "valid" / "labels"),
    "test":  (DATASET_DIR / "test"  / "images", DATASET_DIR / "test"  / "labels"),
}

# Training dirs are always needed for the reference distribution in section 9.
TRAIN_IMG_DIR = DATASET_DIR / "train" / "images"
TRAIN_LBL_DIR = DATASET_DIR / "train" / "labels"

YOLO_WEIGHTS = ROOT / "models" / "runs" / "bib-yolov8n" / "weights" / "best.pt"

# EMA checkpoint from rfdetr-nano-map50 run — best val ema_mAP50 (epoch 11, 0.970).
# Falls back to the original run if not present.
_ema_weights  = ROOT / "models" / "runs" / "rfdetr-nano-map50" / "checkpoint_best_ema.pth"
_orig_weights = ROOT / "models" / "runs" / "rfdetr-nano"       / "checkpoint_best_total.pth"
RFDETR_WEIGHTS = _ema_weights if _ema_weights.exists() else _orig_weights

YOLO_CONF   = 0.25
RFDETR_CONF = 0.50
IOU_THR     = 0.50

ARTIFACTS = Path("artifacts/image-analysis")
ARTIFACTS.mkdir(parents=True, exist_ok=True)

print(f"YOLO weights   : {YOLO_WEIGHTS}")
print(f"RF-DETR weights: {RFDETR_WEIGHTS}")
print(f"Eval splits    : {EVAL_SPLITS}")
for s in EVAL_SPLITS:
    img_dir, _ = SPLIT_DIRS[s]
    print(f"  {s:5s} — {len(list(img_dir.glob('*')))} images")

## Utilities

In [ ]:
def parse_gt(label_path: Path, img_w: int, img_h: int) -> np.ndarray:
    """Parse YOLO-format label file → xyxy boxes (absolute pixels)."""
    boxes = []
    if label_path.exists():
        for line in label_path.read_text().strip().splitlines():
            parts = line.split()
            if len(parts) < 5:
                continue
            _, cx, cy, bw, bh = map(float, parts[:5])
            boxes.append([
                (cx - bw / 2) * img_w, (cy - bh / 2) * img_h,
                (cx + bw / 2) * img_w, (cy + bh / 2) * img_h,
            ])
    return np.array(boxes, dtype=float) if boxes else np.zeros((0, 4))


def box_iou(b1: np.ndarray, b2: np.ndarray) -> float:
    ix1, iy1 = max(b1[0], b2[0]), max(b1[1], b2[1])
    ix2, iy2 = min(b1[2], b2[2]), min(b1[3], b2[3])
    inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    a1 = (b1[2] - b1[0]) * (b1[3] - b1[1])
    a2 = (b2[2] - b2[0]) * (b2[3] - b2[1])
    union = a1 + a2 - inter
    return float(inter / union) if union > 0 else 0.0


def match_predictions(
    pred_boxes: np.ndarray, pred_confs: np.ndarray, gt_boxes: np.ndarray, iou_thr: float = IOU_THR
) -> dict:
    """Greedy IoU matching. Returns {gt_idx: (conf, iou)} for matched bibs; absent = FN."""
    matched_gt: dict = {}
    order = np.argsort(-pred_confs)  # highest confidence first
    for pi in order:
        pb, pc = pred_boxes[pi], pred_confs[pi]
        best_iou, best_gi = 0.0, -1
        for gi, gb in enumerate(gt_boxes):
            if gi in matched_gt:
                continue
            iou = box_iou(pb, gb)
            if iou > best_iou:
                best_iou, best_gi = iou, gi
        if best_iou >= iou_thr and best_gi >= 0:
            matched_gt[best_gi] = (float(pc), best_iou)
    return matched_gt


def bib_crop(img_np: np.ndarray, box: np.ndarray, pad: int = 0) -> np.ndarray:
    h, w = img_np.shape[:2]
    x1 = max(0, int(box[0]) - pad)
    y1 = max(0, int(box[1]) - pad)
    x2 = min(w, int(box[2]) + pad)
    y2 = min(h, int(box[3]) + pad)
    return img_np[y1:y2, x1:x2]


def crop_attributes(crop_rgb: np.ndarray) -> tuple:
    """Return (brightness, contrast, sharpness) for an RGB crop."""
    gray = cv2.cvtColor(crop_rgb, cv2.COLOR_RGB2GRAY).astype(np.float64)
    brightness = float(gray.mean())
    contrast   = float(gray.std())
    sharpness  = float(cv2.Laplacian(gray, cv2.CV_64F).var())
    return brightness, contrast, sharpness


def image_attributes(img_np: np.ndarray) -> tuple:
    """Return (brightness, sharpness) for a full image."""
    gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY).astype(np.float64)
    return float(gray.mean()), float(cv2.Laplacian(gray, cv2.CV_64F).var())


print("Utilities defined.")

## Load Models

In [ ]:
yolo_model   = YOLO(YOLO_WEIGHTS)
rfdetr_model = RFDETRNano(pretrain_weights=str(RFDETR_WEIGHTS))
print("Both models loaded.")

## Build Per-Bib Dataset

One row per ground-truth bib across all evaluated splits. Columns include:
- **Split:** which dataset split (train / val / test)
- **Geometry:** size, aspect ratio, position in frame
- **Local image quality:** brightness, contrast, sharpness of the bib crop
- **Image-level:** overall image brightness, sharpness, crowd density
- **Outcomes:** whether each model detected the bib, and with what confidence

Evaluating on training data tells you where the model struggles on *seen* examples
(a structural weakness); val/test failures reveal generalization gaps.

In [ ]:
rows = []

for split in EVAL_SPLITS:
    img_dir, lbl_dir = SPLIT_DIRS[split]
    split_images = sorted(img_dir.glob("*.*"))
    print(f"Processing {split} ({len(split_images)} images)…")

    for img_path in split_images:
        img_pil = Image.open(img_path).convert("RGB")
        img_np  = np.array(img_pil)
        img_h, img_w = img_np.shape[:2]

        img_brightness, img_sharpness = image_attributes(img_np)

        lbl_path = lbl_dir / (img_path.stem + ".txt")
        gt_boxes = parse_gt(lbl_path, img_w, img_h)
        if len(gt_boxes) == 0:
            continue

        # YOLO inference
        res = yolo_model.predict(str(img_path), conf=YOLO_CONF, device=DEVICE, verbose=False)
        yolo_boxes = res[0].boxes.xyxy.cpu().numpy() if len(res[0].boxes) else np.zeros((0, 4))
        yolo_confs = res[0].boxes.conf.cpu().numpy() if len(res[0].boxes) else np.zeros(0)

        # RF-DETR inference
        rfdet = rfdetr_model.predict(img_pil, threshold=RFDETR_CONF)
        rfdetr_boxes = rfdet.xyxy       if len(rfdet) else np.zeros((0, 4))
        rfdetr_confs = rfdet.confidence if rfdet.confidence is not None else np.zeros(0)

        yolo_matched   = match_predictions(yolo_boxes,   yolo_confs,   gt_boxes)
        rfdetr_matched = match_predictions(rfdetr_boxes, rfdetr_confs, gt_boxes)

        for gi, gb in enumerate(gt_boxes):
            bw = gb[2] - gb[0]
            bh = gb[3] - gb[1]
            if bw <= 0 or bh <= 0:
                continue
            cx = (gb[0] + gb[2]) / 2
            cy = (gb[1] + gb[3]) / 2

            crop = bib_crop(img_np, gb)
            if crop.size == 0:
                continue
            loc_brightness, loc_contrast, loc_sharpness = crop_attributes(crop)

            yolo_conf   = yolo_matched[gi][0]   if gi in yolo_matched   else np.nan
            rfdetr_conf = rfdetr_matched[gi][0] if gi in rfdetr_matched else np.nan

            rows.append({
                "split":          split,
                "image":          img_path.name,
                "gi":             gi,
                # geometry
                "size_px2":       bw * bh,
                "size_frac":      (bw * bh) / (img_w * img_h),
                "aspect_ratio":   bw / bh,
                "cx_norm":        cx / img_w,
                "cy_norm":        cy / img_h,
                "dist_center":    np.sqrt(((cx / img_w) - 0.5) ** 2 + ((cy / img_h) - 0.5) ** 2),
                # local quality
                "local_brightness": loc_brightness,
                "local_contrast":   loc_contrast,
                "local_sharpness":  loc_sharpness,
                # image-level
                "img_brightness": img_brightness,
                "img_sharpness":  img_sharpness,
                "n_bibs":         len(gt_boxes),
                # outcomes
                "yolo_detected":   gi in yolo_matched,
                "yolo_conf":       yolo_conf,
                "rfdetr_detected": gi in rfdetr_matched,
                "rfdetr_conf":     rfdetr_conf,
            })

bib_df = pd.DataFrame(rows)

print(f"\nTotal GT bibs : {len(bib_df)}")
for split in EVAL_SPLITS:
    sub = bib_df[bib_df["split"] == split]
    print(f"  {split:5s} — {len(sub):3d} bibs | "
          f"YOLO TP/FN: {sub['yolo_detected'].sum()}/{(~sub['yolo_detected']).sum()} | "
          f"RF-DETR TP/FN: {sub['rfdetr_detected'].sum()}/{(~sub['rfdetr_detected']).sum()}")
bib_df.head(3)

## Plotting helper

In [ ]:
MODEL_META = {
    "yolo":   {"label": "YOLOv8n",      "col": "yolo_detected",   "conf_col": "yolo_conf",   "color": "steelblue"},
    "rfdetr": {"label": "RF-DETR Nano", "col": "rfdetr_detected", "conf_col": "rfdetr_conf", "color": "darkorange"},
}


def outcome_hist(ax, df, attr, model_key, xlabel, bins=15, log_x=False):
    """Histogram of `attr` split by detected vs missed for one model."""
    meta = MODEL_META[model_key]
    tp = df[df[meta["col"]]][attr].dropna()
    fn = df[~df[meta["col"]]][attr].dropna()

    all_vals = pd.concat([tp, fn])
    if log_x and all_vals.min() > 0:
        bins = np.logspace(np.log10(all_vals.min()), np.log10(all_vals.max()), bins)
    else:
        bins = np.linspace(all_vals.min(), all_vals.max(), bins)

    ax.hist(tp, bins=bins, alpha=0.6, color=meta["color"], label=f"Detected (n={len(tp)})")
    ax.hist(fn, bins=bins, alpha=0.85, color="tomato",     label=f"Missed (n={len(fn)})")
    if log_x:
        ax.set_xscale("log")
    ax.set_title(meta["label"])
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Count")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)


def two_model_fig(df, attr, xlabel, log_x=False, bins=15, filename=None):
    """Side-by-side outcome histograms for both models."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)
    for ax, key in zip(axes, MODEL_META):
        outcome_hist(ax, df, attr, key, xlabel, bins=bins, log_x=log_x)
    plt.suptitle(f"{xlabel} — detected vs. missed", fontsize=13)
    plt.tight_layout()
    if filename:
        plt.savefig(ARTIFACTS / filename, dpi=150, bbox_inches="tight")
        print(f"Saved {ARTIFACTS / filename}")
    plt.show()


print("Helper functions defined.")

## 1. Bib Size

Baseline analysis: do both models struggle with small bibs?

In [ ]:
two_model_fig(bib_df, "size_px2", "Bib area (px²)", log_x=True, bins=20, filename="attr_size.png")

for key, meta in MODEL_META.items():
    tp = bib_df[bib_df[meta["col"]]]
    fn = bib_df[~bib_df[meta["col"]]]
    print(f"{meta['label']:16s}  TP median size: {tp['size_px2'].median():.0f} px²  "
          f"FN median size: {fn['size_px2'].median():.0f} px²  "
          f"(ratio {fn['size_px2'].median() / tp['size_px2'].median():.2f}x)")

## 2. Local Brightness

Mean pixel intensity of the bib crop (0 = black, 255 = white).
Dark bibs (backlit runners, shadowed areas) may be systematically missed.

In [ ]:
two_model_fig(bib_df, "local_brightness", "Local brightness (mean gray, 0–255)",
              filename="attr_local_brightness.png")

for key, meta in MODEL_META.items():
    tp = bib_df[bib_df[meta["col"]]]
    fn = bib_df[~bib_df[meta["col"]]]
    print(f"{meta['label']:16s}  TP brightness: {tp['local_brightness'].mean():.1f}  "
          f"FN brightness: {fn['local_brightness'].mean():.1f}")

## 3. Local Contrast

Standard deviation of pixel values in the bib crop.
Low-contrast bibs (white number on white shirt, or uniform dark bib) are harder to detect.

In [ ]:
two_model_fig(bib_df, "local_contrast", "Local contrast (std dev of gray)",
              filename="attr_local_contrast.png")

for key, meta in MODEL_META.items():
    tp = bib_df[bib_df[meta["col"]]]
    fn = bib_df[~bib_df[meta["col"]]]
    print(f"{meta['label']:16s}  TP contrast: {tp['local_contrast'].mean():.1f}  "
          f"FN contrast: {fn['local_contrast'].mean():.1f}")

## 4. Local Sharpness

Variance of the Laplacian of the bib crop — higher means sharper.
Motion blur from running is a common failure mode in race photography.

In [ ]:
two_model_fig(bib_df, "local_sharpness", "Local sharpness (Laplacian variance, log)",
              log_x=True, bins=20, filename="attr_local_sharpness.png")

for key, meta in MODEL_META.items():
    tp = bib_df[bib_df[meta["col"]]]
    fn = bib_df[~bib_df[meta["col"]]]
    print(f"{meta['label']:16s}  TP sharpness median: {tp['local_sharpness'].median():.1f}  "
          f"FN sharpness median: {fn['local_sharpness'].median():.1f}")

## 5. Position in Frame

2D scatter of bib center positions (normalized 0–1), colored by outcome.
Edge-of-frame or top/bottom bibs may be systematically harder.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (key, meta) in zip(axes, MODEL_META.items()):
    tp = bib_df[bib_df[meta["col"]]]
    fn = bib_df[~bib_df[meta["col"]]]

    ax.scatter(tp["cx_norm"], tp["cy_norm"], c=meta["color"], alpha=0.6,
               s=60, label=f"Detected (n={len(tp)})", zorder=3)
    ax.scatter(fn["cx_norm"], fn["cy_norm"], c="tomato", marker="X",
               s=120, label=f"Missed (n={len(fn)})", zorder=4, edgecolors="black", linewidths=0.5)

    # Frame reference
    ax.axhline(0.5, color="gray", linestyle=":", linewidth=0.8)
    ax.axvline(0.5, color="gray", linestyle=":", linewidth=0.8)
    ax.set_xlim(0, 1)
    ax.set_ylim(1, 0)  # y=0 at top, matching image coordinates
    ax.set_xlabel("Horizontal position (0=left, 1=right)")
    ax.set_ylabel("Vertical position (0=top, 1=bottom)")
    ax.set_title(meta["label"])
    ax.legend(fontsize=8)
    ax.set_aspect("equal")
    ax.grid(alpha=0.2)

plt.suptitle("Bib position in frame — detected vs. missed", fontsize=13)
plt.tight_layout()
plt.savefig(ARTIFACTS / "attr_position.png", dpi=150, bbox_inches="tight")
print(f"Saved {ARTIFACTS / 'attr_position.png'}")
plt.show()

## 6. Aspect Ratio

Width / height of the bib box. Strongly non-square bibs may indicate partial occlusion or an unusual viewing angle.

In [ ]:
two_model_fig(bib_df, "aspect_ratio", "Bib aspect ratio (width / height)",
              filename="attr_aspect_ratio.png")

## 7. Confidence vs. Bib Attributes

For bibs that **were** detected, does confidence correlate with any attribute?
A strong correlation means bibs with that attribute are at risk of being missed
if the detection threshold is raised — even if they aren't missed here.

In [ ]:
attrs = [
    ("size_px2",        "Bib area (px²)",                 True),
    ("local_brightness","Local brightness",               False),
    ("local_contrast",  "Local contrast",                 False),
    ("local_sharpness", "Local sharpness (Laplacian var)", True),
]

for key, meta in MODEL_META.items():
    detected = bib_df[bib_df[meta["col"]]].copy()
    conf_col  = meta["conf_col"]

    fig, axes = plt.subplots(1, len(attrs), figsize=(16, 4))
    for ax, (attr, xlabel, log_x) in zip(axes, attrs):
        x = detected[attr].values
        y = detected[conf_col].values
        ax.scatter(x, y, alpha=0.6, color=meta["color"], s=50)
        if len(x) > 1:
            # Pearson r on log(x) if log scale
            xv = np.log10(x + 1e-9) if log_x else x
            r = np.corrcoef(xv, y)[0, 1]
            ax.set_title(f"{xlabel}\nr = {r:.2f}", fontsize=9)
        else:
            ax.set_title(xlabel, fontsize=9)
        if log_x:
            ax.set_xscale("log")
        ax.set_xlabel(xlabel, fontsize=8)
        ax.set_ylabel("Confidence", fontsize=8)
        ax.set_ylim(0, 1.05)
        ax.grid(alpha=0.3)

    plt.suptitle(f"{meta['label']} — confidence vs. bib attributes (detected bibs only)", fontsize=12)
    plt.tight_layout()
    fname = f"attr_conf_scatter_{key}.png"
    plt.savefig(ARTIFACTS / fname, dpi=150, bbox_inches="tight")
    print(f"Saved {ARTIFACTS / fname}")
    plt.show()

## 8. Image-Level Analysis

Per-image precision and recall, correlated with crowd density and overall image quality.

In [ ]:
img_rows = []
for img_name, grp in bib_df.groupby("image"):
    n_gt = len(grp)
    img_rows.append({
        "image":          img_name,
        "n_bibs":         n_gt,
        "img_brightness": grp["img_brightness"].iloc[0],
        "img_sharpness":  grp["img_sharpness"].iloc[0],
        "yolo_recall":    grp["yolo_detected"].mean(),
        "rfdetr_recall":  grp["rfdetr_detected"].mean(),
        "yolo_fn":        (~grp["yolo_detected"]).sum(),
        "rfdetr_fn":      (~grp["rfdetr_detected"]).sum(),
    })
img_df = pd.DataFrame(img_rows)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

img_attrs = [
    ("n_bibs",        "Bibs per image (crowd density)", False),
    ("img_brightness","Image brightness",               False),
    ("img_sharpness", "Image sharpness (Laplacian var)", True),
]

for col, (attr, xlabel, log_x) in enumerate(img_attrs):
    for row, (key, meta) in enumerate(MODEL_META.items()):
        ax = axes[row][col]
        x = img_df[attr].values
        y = img_df[f"{key}_recall"].values
        # size = number of FNs (bigger dot = more misses)
        fn_counts = img_df[f"{key}_fn"].values
        sizes = 40 + fn_counts * 60
        sc = ax.scatter(x, y, c=fn_counts, cmap="Reds", s=sizes, alpha=0.8,
                        vmin=0, edgecolors="gray", linewidths=0.5)
        plt.colorbar(sc, ax=ax, label="FN count")
        if log_x:
            ax.set_xscale("log")
        ax.set_xlabel(xlabel, fontsize=8)
        ax.set_ylabel("Per-image recall", fontsize=8)
        ax.set_ylim(-0.05, 1.1)
        ax.set_title(f"{meta['label']}", fontsize=9)
        ax.axhline(1.0, color="green", linestyle="--", linewidth=0.8, alpha=0.5)
        ax.grid(alpha=0.2)

plt.suptitle("Per-image recall vs. image attributes (dot size ∝ FN count)", fontsize=13)
plt.tight_layout()
plt.savefig(ARTIFACTS / "image_level_analysis.png", dpi=150, bbox_inches="tight")
print(f"Saved {ARTIFACTS / 'image_level_analysis.png'}")
plt.show()

print("\nImages with any FN (either model):")
any_fn = img_df[(img_df["yolo_fn"] > 0) | (img_df["rfdetr_fn"] > 0)][
    ["image", "n_bibs", "img_brightness", "img_sharpness", "yolo_fn", "rfdetr_fn"]
].sort_values("rfdetr_fn", ascending=False)
print(any_fn.to_string(index=False))

## 9. Training Data Distribution vs. Failures

Compare the attribute distribution of **all training bibs** (the reference) against
**bibs missed by at least one model** across all evaluated splits.

A gap between these distributions reveals underrepresented conditions — where to focus
future data collection.

**Interpreting by split:**
- FNs from **test/val** = generalization gaps — conditions underrepresented in training
- FNs from **train** (if included) = structural weaknesses — conditions the model
  can't learn even with exposure; no amount of extra data fixes these alone

In [ ]:
# Build training bib attribute dataset (GT attributes only — no inference needed)
train_rows = []
for lbl_path in sorted(TRAIN_LBL_DIR.glob("*.txt")):
    # Find corresponding image (any extension)
    candidates = list(TRAIN_IMG_DIR.glob(lbl_path.stem + ".*"))
    if not candidates:
        continue
    img_path = candidates[0]
    img_np = np.array(Image.open(img_path).convert("RGB"))
    img_h, img_w = img_np.shape[:2]
    gt_boxes = parse_gt(lbl_path, img_w, img_h)

    for gb in gt_boxes:
        bw = gb[2] - gb[0]
        bh = gb[3] - gb[1]
        if bw <= 0 or bh <= 0:
            continue
        crop = bib_crop(img_np, gb)
        if crop.size == 0:
            continue
        loc_brightness, loc_contrast, loc_sharpness = crop_attributes(crop)
        train_rows.append({
            "size_px2":        bw * bh,
            "local_brightness": loc_brightness,
            "local_contrast":   loc_contrast,
            "local_sharpness":  loc_sharpness,
            "aspect_ratio":    bw / bh,
        })

train_df = pd.DataFrame(train_rows)
print(f"Training GT bibs: {len(train_df)}")
train_df.describe().round(1)

In [ ]:
compare_attrs = [
    ("size_px2",         "Bib area (px²)",                True),
    ("local_brightness", "Local brightness (mean gray)",   False),
    ("local_contrast",   "Local contrast (std gray)",      False),
    ("local_sharpness",  "Local sharpness (Laplacian var)",True),
]

# FN bibs per split (missed by at least one model)
fn_by_split = {
    s: bib_df[(bib_df["split"] == s) & (~bib_df["yolo_detected"] | ~bib_df["rfdetr_detected"])]
    for s in EVAL_SPLITS
}
any_fn_bibs = pd.concat(fn_by_split.values())
print(f"Missed by either model — total: {len(any_fn_bibs)}")
for s in EVAL_SPLITS:
    sub = fn_by_split[s]
    print(f"  {s:5s}: {len(sub)} FN bibs")

# Split colors for the density overlay
split_colors = {"train": "seagreen", "val": "mediumpurple", "test": "tomato"}

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

for ax, (attr, xlabel, log_x) in zip(axes.flatten(), compare_attrs):
    train_vals = train_df[attr].dropna()

    all_vals = pd.concat([train_vals] + [fn_by_split[s][attr].dropna() for s in EVAL_SPLITS])
    lo = max(all_vals.min(), 1e-6)
    hi = all_vals.max()
    bins = np.logspace(np.log10(lo), np.log10(hi), 25) if log_x else np.linspace(lo, hi, 25)

    ax.hist(train_vals, bins=bins, density=True, alpha=0.45, color="steelblue",
            label=f"Train GT (n={len(train_vals)})")

    for s in EVAL_SPLITS:
        fn_vals = fn_by_split[s][attr].dropna()
        if len(fn_vals) > 0:
            ax.hist(fn_vals, bins=bins, density=True, alpha=0.65,
                    color=split_colors.get(s, "gray"),
                    label=f"{s} FN — any model (n={len(fn_vals)})",
                    histtype="step", linewidth=2)

    if log_x:
        ax.set_xscale("log")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Density")
    ax.set_title(xlabel)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle(
    "Training data distribution vs. missed detections by split\n"
    "Gaps show where to collect more training examples",
    fontsize=13,
)
plt.tight_layout()
plt.savefig(ARTIFACTS / "training_vs_failures.png", dpi=150, bbox_inches="tight")
print(f"\nSaved {ARTIFACTS / 'training_vs_failures.png'}")
plt.show()

# Summary stats
print("\nMedian attribute values — training GT vs. FNs per split:")
for attr, xlabel, _ in compare_attrs:
    print(f"\n  {xlabel}")
    print(f"    Train GT : {train_df[attr].median():.1f}")
    for s in EVAL_SPLITS:
        fn_vals = fn_by_split[s][attr].dropna()
        if len(fn_vals):
            ratio = fn_vals.median() / train_df[attr].median()
            print(f"    {s:5s} FN : {fn_vals.median():.1f}  ({ratio:.2f}x train median)")

## 10. Summary — Where to Focus Data Collection

For each attribute, compare the median value for detected bibs vs. missed bibs.
A large gap = a meaningful signal about what conditions are underrepresented.

In [ ]:
stat_attrs = [
    ("size_px2",         "Bib area (px²)"),
    ("local_brightness", "Local brightness"),
    ("local_contrast",   "Local contrast"),
    ("local_sharpness",  "Local sharpness"),
    ("dist_center",      "Distance from frame center"),
    ("aspect_ratio",     "Aspect ratio"),
]

# --- Overall summary (all splits combined) ---
summary_rows = []
for attr, label in stat_attrs:
    row = {"Attribute": label}
    for key, meta in MODEL_META.items():
        tp = bib_df[bib_df[meta["col"]]][attr].dropna()
        fn = bib_df[~bib_df[meta["col"]]][attr].dropna()
        if len(fn) == 0:
            row[f"{meta['label']} TP med"] = f"{tp.median():.1f}"
            row[f"{meta['label']} FN med"] = "n/a"
            row[f"{meta['label']} FN/TP"]  = "—"
        else:
            ratio = fn.median() / tp.median() if tp.median() != 0 else float("inf")
            row[f"{meta['label']} TP med"] = f"{tp.median():.1f}"
            row[f"{meta['label']} FN med"] = f"{fn.median():.1f}"
            row[f"{meta['label']} FN/TP"]  = f"{ratio:.2f}x"
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).set_index("Attribute")
print("Overall attribute summary (all splits) — ratio < 1 = FN bibs have lower values")
display(summary_df)

# --- Per-split FN counts by model ---
print("\nFalse negatives by split and model:")
split_fn_rows = []
for s in EVAL_SPLITS:
    sub = bib_df[bib_df["split"] == s]
    split_fn_rows.append({
        "Split":         s,
        "GT bibs":       len(sub),
        "YOLO FN":       int((~sub["yolo_detected"]).sum()),
        "YOLO recall":   f"{sub['yolo_detected'].mean():.3f}",
        "RF-DETR FN":    int((~sub["rfdetr_detected"]).sum()),
        "RF-DETR recall":f"{sub['rfdetr_detected'].mean():.3f}",
        "FN either":     int((~sub["yolo_detected"] | ~sub["rfdetr_detected"]).sum()),
        "FN both":       int((~sub["yolo_detected"] & ~sub["rfdetr_detected"]).sum()),
    })
display(pd.DataFrame(split_fn_rows).set_index("Split"))

# --- Training data gap summary ---
print("\nTraining data gap summary (train GT vs. test FN bibs):")
test_fn = bib_df[(bib_df["split"] == "test") & (~bib_df["yolo_detected"] | ~bib_df["rfdetr_detected"])]
if len(test_fn):
    print(f"  Train median size  : {train_df['size_px2'].median():.0f} px²")
    print(f"  Test FN median size: {test_fn['size_px2'].median():.0f} px²  "
          f"({train_df['size_px2'].median() / test_fn['size_px2'].median():.1f}x larger in training)")
    print(f"  Train median brightness  : {train_df['local_brightness'].median():.1f}")
    print(f"  Test FN median brightness: {test_fn['local_brightness'].median():.1f}")